In [29]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from toolsos.huisstijl.colors import get_os_colors
from graphs import simple_barh, simple_line, multiple_line
import pickle
from itertools import cycle

In [32]:
no_onbewerkt = pd.read_excel('../../04 tabellen/02 tabellen noord/tabel alle data long Aanpak Noord 2026_05.xlsx')

In [28]:
type(no_onbewerkt.temporal_date[0])


numpy.int64

In [31]:
def bewerk_dataframe(df_onbewerkt):
    df = df_onbewerkt.loc[
        (df_onbewerkt["spatial_type"].isin(["stadsdelen", "gemeente"]))  &
        (df_onbewerkt['tweedeling_def'] == 'totaal')
    ].copy()

    df["temporal_date"] = pd.to_datetime(df["temporal_date"], format="%Y")
    df["temporal_date"] = df["temporal_date"].dt.year.astype(str)

    #df = df.sort_values(["spatial_name", "measure", "temporal_date"])

    return df

sd = bewerk_dataframe(no_onbewerkt)

OutOfBoundsDatetime: Out of bounds nanosecond timestamp: 4054, at position 9. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [16]:
indicatoren = sd['measure'].unique()
kernindicatoren = sd.loc[sd['kernindicator_noord']==1]['measure'].unique()
indicatoren = [indicator for indicator in indicatoren if indicator not in kernindicatoren]

kleuren = get_os_colors(type='discreet', kleur='discreet (1-9)', aantal='9')  # Lijst met kleuren
kleur_mapping = {kernindicator: kleur for kernindicator, kleur in zip(kernindicatoren, cycle(kleuren))}


#### Bars

In [17]:
for indicator in kernindicatoren:
    df = sd[(sd['measure']==indicator) & (sd['spatial_type']=='stadsdelen')]
    if df['value'].notna().any() and '_r' in indicator:
        
        kleur = kleur_mapping.get(indicator, '#004699') # Zoek de kleur op in de mapping

        simple_barh(
            df=df,
            x_col='temporal_date',
            y_col='value',
            y_min=0,
            y_max=10,
            color_func=kleur,
            output_path=f'../../05 reports/Noord/andere indicatoren/{indicator}.png' 
        )

#### Multiple Line Ams en SD

In [18]:
def multiple_line_perc_figuur(indicator_list, output_folder):
    for indicator in indicator_list:
        
        df = sd[sd['measure'] == indicator]

        if df['value'].notna().any():

            kleur = kleur_mapping.get(indicator, '#004699')  # Zoek de kleur op in de mapping

            # Controleer of '_p' of '_r' in de indicatornaam staat en pas y-as daarop aan
            if '_p' in indicator:
                y_min = 0
                y_max = 100 if df['value'].max() > 20 else 30
                is_percentage = True
            elif '_r' in indicator:
                y_min = 1
                y_max = 10
                is_percentage = False
            else:
                # Gebruik standaardwaarden als geen '_p' of '_r' aanwezig is
                y_min = None
                y_max = None
                is_percentage = False 
                                    
            multiple_line(
                df=df,
                x_col='temporal_date',
                y_col='value',
                main_area='Amsterdam', 
                y_min= y_min,
                y_max= y_max,
                color_func=kleur,  # Gebruik de kleur uit de mapping
                is_percentage=is_percentage,
                output_path=f'../../05 reports/Noord/{output_folder}/{indicator}.png'
                )

# Verwerk indicatoren
multiple_line_perc_figuur(indicatoren, 'andere indicatoren')

# Verwerk kernindicatoren
multiple_line_perc_figuur(kernindicatoren, 'kernindicatoren')

In [19]:
sd.to_excel('../../05 reports/Noord/data_indicatoren.xlsx')